In [ ]:
import differences as df
import matplotlib.pyplot as plt
import numpy as np
import mibitrans as mbt

### Sensitivity analysis of differences between Mibitrans, Anatrans and Bioscreen models

This notebook shows the influence of dispersivity on the error size of the approximate analytical solutions of Anatrans and Bioscreen compared to the exact solution of Mibitrans.

In [ ]:
def sensitivity_models(
        hydrological_parameters,
        attenuation_parameters,
        source_parameters,
        model_parameters,
        x_dispersivity,
        y_dispersivity,
        z_dispersivity,
):
    """Repeatedly run models with different dispersivities for sensitivity analysis."""
    list_mibitrans = [[[0]*len(x_dispersivity)
                        for _ in range(len(y_dispersivity))]
                        for _ in range(len(z_dispersivity))]
    list_anatrans = [[[0]*len(x_dispersivity)
                        for _ in range(len(y_dispersivity))]
                        for _ in range(len(z_dispersivity))]
    list_bioscreen = [[[0]*len(x_dispersivity)
                        for _ in range(len(y_dispersivity))]
                        for _ in range(len(z_dispersivity))]

    mibitrans_object = mbt.Mibitrans(hydrological_parameters,
                                     attenuation_parameters,
                                     source_parameters,
                                     model_parameters)

    anatrans_object = mbt.Anatrans(hydrological_parameters,
                                   attenuation_parameters,
                                   source_parameters,
                                   model_parameters)

    bioscreen_object = mbt.Bioscreen(hydrological_parameters,
                                     attenuation_parameters,
                                     source_parameters,
                                     model_parameters)

    velocity = hydrological_parameters.velocity
    porosity = hydrological_parameters.porosity
    diffusion = hydrological_parameters.diffusion

    for i in range(len(x_dispersivity)):
        for j in range(len(y_dispersivity)):
            for k in range(len(z_dispersivity)):
                hydro_dispersivity = mbt.HydrologicalParameters(
                    velocity=velocity,
                    porosity=porosity,
                    alpha_x=x_dispersivity[i],
                    alpha_y=y_dispersivity[j],
                    alpha_z=z_dispersivity[k],
                    diffusion=diffusion,
                )
                print(i,j,k) # Print to track progress
                mibitrans_object.hydrological_parameters = hydro_dispersivity
                list_mibitrans[k][j][i] = mibitrans_object.run()
                anatrans_object.hydrological_parameters = hydro_dispersivity
                list_anatrans[k][j][i] = anatrans_object.run()
                bioscreen_object.hydrological_parameters = hydro_dispersivity
                list_bioscreen[k][j][i] = bioscreen_object.run()

    return list_mibitrans, list_anatrans, list_bioscreen

In [ ]:
def comparison_plot(
    list_mibitrans, list_compare, x_dispersivity, y_dispersivity, z_dispersivity,
    time_index=None,
    difference_method="absolute",
    cutoff=1e-5,
    relative_concentration=True,
    relative_diff_diff=False,
    as_percentage=False,
    colorbar_label=None,
):
    """Plot difference between models, averaged over model domain, for all combinations of dispersivity."""
    ##### Plotting preferences #####
    value_decimals = 4
    figuresize = (7,5)
    colormap = "viridis"
    second_y_axis_location = -0.24
    x_label = r"Longitudinal dispersivity ($\alpha_x$) [m]"
    first_y_label = r"Transverse vertical dispersivity $\alpha_z$ [m]"
    second_y_label = r"Transverse horizontal dispersivity $\alpha_y$ [m]"
    #################################
    if time_index is None:
        time_index =  len(list_mibitrans[0][0][0].t)

    if relative_concentration:
        c_mode = "relative_cxyt"
    else:
        c_mode ="cxyt"

    difference_array = np.zeros((len(y_dispersivity) * len(z_dispersivity), len(x_dispersivity)))
    if difference_method not in ["relative", "rmse", "absolute", "sum_mean"]:
        print("Difference method not recognized, defaulting to absolute")

    for i in range(len(x_dispersivity)):
        for j in range(len(y_dispersivity)):
            for k in range(len(z_dispersivity)):
                if difference_method == "relative":
                    difference_array[len(z_dispersivity)*j+k,i] = df.mean_relative_difference(
                        getattr(list_mibitrans[k][j][i], c_mode)[:time_index,:,:],
                        getattr(list_compare[k][j][i], c_mode)[:time_index,:,:],
                        concentration_cutoff=cutoff
                    )
                elif difference_method == "rmse":
                    difference_array[len(z_dispersivity)*j+k,i] = df.rmse(
                        getattr(list_mibitrans[k][j][i], c_mode)[:time_index,:,:],
                        getattr(list_compare[k][j][i], c_mode)[:time_index,:,:],
                        concentration_cutoff=cutoff
                    )
                elif difference_method == "sum_mean":
                    difference_array[len(z_dispersivity)*j+k,i] = df.mean_sum_absolute_difference(
                        getattr(list_mibitrans[k][j][i], c_mode)[:time_index,:,:],
                        getattr(list_compare[k][j][i], c_mode)[:time_index,:,:],
                        concentration_cutoff=cutoff
                    )
                else:
                    difference_array[len(z_dispersivity)*j+k,i] = df.mean_absolute_difference(
                        getattr(list_mibitrans[k][j][i], c_mode)[:time_index,:,:],
                        getattr(list_compare[k][j][i], c_mode)[:time_index,:,:],
                        concentration_cutoff=cutoff
                    )

    if as_percentage:
        difference_array = difference_array * 100
    if relative_diff_diff:
        difference_array = difference_array / np.max(difference_array)

    z_ticks = []
    for i in range(len(y_dispersivity)):
        for j in range(len(z_dispersivity)):
            z_ticks.append(f"{z_dispersivity[j]}")

    fig, ax = plt.subplots(figsize=figuresize)
    imshow_aspect = str(1/len(z_dispersivity))
    im = ax.imshow(
        difference_array,
        cmap=colormap,
        aspect=imshow_aspect
    )
    ax2 = ax.secondary_yaxis(second_y_axis_location)
    middle_z_dispersivity_axis = len(z_dispersivity)/2-0.5

    second_y_tick_location = np.linspace(
        middle_z_dispersivity_axis,
        (len(y_dispersivity)-1)*len(z_dispersivity) + middle_z_dispersivity_axis,
        len(y_dispersivity))

    ax.set_xticks(range(len(x_dispersivity)), labels=x_dispersivity)
    ax.set_yticks(range(len(z_ticks)), labels=z_ticks)
    ax2.set_yticks(second_y_tick_location, labels=y_dispersivity)

    ax.hlines(second_y_tick_location[:-1]+len(z_dispersivity)/2,
              -0.5,
              len(x_dispersivity)-0.5,
              color="white",
              lw=2)
    ax.vlines(
        np.arange(0.5,len(x_dispersivity)-1,1),
        -0.5, len(y_dispersivity)*len(z_dispersivity)-0.5,
        color="white",
        lw=2
    )

    for i in range(len(x_dispersivity)):
        for j in range(len(y_dispersivity)*len(z_dispersivity)):
            ax.text(
                i, j, np.round(difference_array[j, i], decimals=value_decimals),
                ha="center",
                va="center",
                color="w"
            )

    ax.set_xlabel(x_label)
    ax.set_ylabel(first_y_label)
    ax2.set_ylabel(second_y_label)
    fig.colorbar(im, label=colorbar_label)
    #plt.savefig("mbt_ana_comp_dual_axis.png", dpi=500, bbox_inches="tight")

Having made the functions performing the model runs and plotting, define transport parameters and disperivity values to vary:

In [ ]:
x_disp = np.array([10,3,1])                   #[m]
y_disp = np.array([0.2,0.02,0.002])           #[m]
z_disp = np.array([0.05, 0.005, 0.0005, 0])   #[m]

att = mbt.AttenuationParameters(
    retardation=1.3,      #[m]
    half_life=5*365     #[days]
)

source = mbt.SourceParameters(
    source_zone_boundary=5,        #[m]
    source_zone_concentration=10,   #[g/m3]
    depth=2,                                  #[m]
    total_mass=np.inf                          #[g]
)

model = mbt.ModelParameters(
    model_length = 400,    #[m]
    model_width = 60,      #[m]
    model_time = 10*365,    #[m]
    dx = 2,                 #[m]
    dy = 0.5,                 #[m]
    dt = 365                #[days]
)

hydro = mbt.HydrologicalParameters(
    velocity=0.1, #[m/d]
    porosity=0.3,  #[-]
    alpha_x=10,     #[m]
    alpha_y=0.02,      #[m]
    alpha_z=0.005,    #[m]
    diffusion=0, #[m2/day]
)

Run the models, output is a nested list, indexed as [z][y][x], in the order of the dispersivity arrays defined above. Thus, in this example, `list[2][0][1]` would give the model results with $\alpha_x = 3m$, $\alpha_y = 0.2m$, $\alpha_z = 0.0005m$.

In [ ]:
list_mbt, list_ana, list_bio = sensitivity_models(hydro, att, source, model, x_disp, y_disp, z_disp)


Then compare the two models using the absolute difference between relative concentrations.

In [ ]:
comparison_plot(
    list_mbt, list_ana, x_disp, y_disp, z_disp,
    cutoff = 0, # Concentration value below which a value will not be included in the average
    relative_concentration = True, # If True, differences are calculated from relative concentrations C/C0.
    difference_method = "absolute", # 'relative', 'absolute' or 'rmse'
    relative_diff_diff = False, # If True, largest value will be set to 1, other values are scaled proportionally
    as_percentage = True, # If True, multiply all values by 100, prevents too much 0's behind the decimal.
    colorbar_label = "mean absolute difference in relative concentration [%]"
)

These results fall within what is expected based on theory, longitudinal dispersivity has a large influence on the deviation of the Anatrans solution from the Mibitrans solution. If transverse horizontal dispersivity is high, transverse vertical dispersivity has a relatively small influence, although its influence increases for lower transverse horizontal dispersivities. taking the average across the entire model domain is not a valid comparison method. This way includes a large amount of 0 values, either because the advective front has not progressed enough, or due to the plume not dispersing as far as the model boundary. An alternative is to set a cutoff value, below which cells are not included in calculation of the mean difference.

In [ ]:
comparison_plot(
    list_mbt, list_ana, x_disp, y_disp, z_disp,
    cutoff = 1e-3,
    relative_concentration = True,
    difference_method = "absolute",
    relative_diff_diff = False,
    as_percentage = True,
    colorbar_label = "mean absolute difference in relative concentration [%]"
)

Although the pattern in effect of longitudinal dispersivity is preserved, the effect of transverse horizontal dispersivity is contrary to the previous plot, and contrary to what is expected from the theory. If the Anatrans model is equal to the Mibitrans model if $\alpha_y = \alpha_z = 0$, one would expect that with $\lim_{\alpha_y, \alpha_z \to 0}$ , differences between the models should approach 0 as well. However, if only $\alpha_y \to 0$, the error introduced by the transverse vertical term still remains. And since the plume spreads little in the transverse horizontal direction, all difference is contained in a small area, leading to a high average. On the contrary, For higher values of transverse horizontal dispersivity, differences might 'add' up to more in total, but since they are spread out over a larger area, they average out to less. Therefore, the values from this method are more reflective of the general behaviour of the 3D-ADE, than of the sensitivity of the approximate solution to dispersivity.

This is visualized in the graphs below.

In [ ]:
time = 5*365
models=[list_mbt[0][-1][0], list_mbt[0][0][0],
        list_ana[0][-1][0], list_ana[0][0][0],
        list_bio[0][-1][0], list_bio[0][0][0]]
colors=["cornflowerblue", "blue",
        "gold", "goldenrod",
        "coral", "red"]
linewidth=2
linestyles=["-", "-", "--", "--", "-.", "-."]
labels=[r"mbt, $\alpha_y=0.002m$", r"mbt, $\alpha_y=0.2m$",
        r"ana, $\alpha_y=0.002m$", r"ana, $\alpha_y=0.2m$",
        r"bio, $\alpha_y=0.002m$", r"bio, $\alpha_y=0.2m$"]

for i, mod in enumerate(models):
    mod.centerline(time=time, color=colors[i], lw=linewidth, linestyle=linestyles[i], label=labels[i])
plt.legend()
plt.xlim(-10, 275)
plt.show()

xpos = 100
for i, mod in enumerate(models):
    mod.transverse(x_position=xpos, time=time, color=colors[i], lw=linewidth, linestyle=linestyles[i], label=labels[i])
plt.legend()
plt.xlim(-20,20)
plt.show()

Another possibility is to use the relative differences between models. However, the smaller values at the plume fringes tend to have larger relative errors, even though the absolute difference at these positions would be too small to be significant for most practical purposes.

In [ ]:
comparison_plot(
    list_mbt, list_ana, x_disp, y_disp, z_disp,
    cutoff = 1e-5,
    relative_concentration = True,
    difference_method = "relative",
    relative_diff_diff = False,
    as_percentage = False,
    colorbar_label = "mean relative difference in concentration"
)

Furthermore, differences now seem highest for the lowest values of $\alpha_z$, even though absolute differences are higher for the higher values of $\alpha_z$, as shown below.

In [ ]:
time = 5*365
models=[list_mbt[-1][0][0], list_mbt[0][0][0],
        list_ana[-1][0][0], list_ana[0][0][0],
        list_bio[-1][0][0], list_bio[0][0][0]]
colors=["cornflowerblue", "blue",
        "gold", "goldenrod",
        "coral", "red"]
linewidth=2
linestyles=["-", "-", "--", "--", "-.", "-."]
labels=[r"mbt, $\alpha_z=0m$", r"mbt, $\alpha_z=0.05m$",
        r"ana, $\alpha_z=0m$", r"ana, $\alpha_z=0.05m$",
        r"bio, $\alpha_z=0m$", r"bio, $\alpha_z=0.05m$"]

for i, mod in enumerate(models):
    mod.centerline(time=time, color=colors[i], lw=linewidth, linestyle=linestyles[i], label=labels[i])
plt.legend()
plt.xlim(-10, 275)
plt.show()

xpos = 100
for i, mod in enumerate(models):
    mod.transverse(x_position=xpos, time=time, color=colors[i], lw=linewidth, linestyle=linestyles[i], label=labels[i])
plt.legend()
plt.xlim(-20,20)
plt.show()

This tenancy of the relative error can be somewhat circumvented by choosing a higher cutoff value, preventing low concentrations from dominating the average. Below, 2% of the source concentration is taken as the cutoff value.

In [ ]:
comparison_plot(
    list_mbt, list_ana, x_disp, y_disp, z_disp,
    cutoff = 2e-2,
    relative_concentration = True,
    difference_method = "relative",
    relative_diff_diff = False,
    as_percentage = False,
    colorbar_label = "mean relative difference in concentration"
)

 However, this makes the choice of the cutoff value rather arbitrary, where how well the resulting values reflect your expectations depend on which value is chosen. Furthermore, for high source concentrations, or higihly toxic contaminants, concentrations below the increased cutoff value might still be significant.

The issues with these methods can be prevented by applying the concentration cutoff in the longitudinal direction only (which eliminates places which the advective front has not yet reached). Then taking the sum of the differences in the transverse horizontal direction, preventing averaging over different domains for different dispersivities. Then taking the mean in the x and t direction in respective order. This last step prevents errors from the earlier time steps, where the advective front has not progressed as far and thus would contribute less to the overall mean, from being underrepresented. This represents differences between the models quite well, however, the values calculated lose their physical meaning. Therefore, it might be better to the differences relative to each other, with 1 being the highest difference among the dispersivities and the rest scaled proportionally (accomplished by setting relative_diff_diff to True). Though this last step is optional. Note how this method is not as sensitive to the choice of cutoff value as the other methods.

In [ ]:
comparison_plot(
    list_mbt, list_ana, x_disp, y_disp, z_disp,
    cutoff = 1e-4,
    relative_concentration = True,
    difference_method = "sum_mean",
    relative_diff_diff = True,
    as_percentage = False,
    colorbar_label = "normalized difference between models"
)

Note: The latter method seems to somewhat overestimate the size in difference for smaller longitudinal dispersivities

In [ ]:
time = 5*365
models=[list_mbt[0][0][0], list_mbt[0][0][-1],
        list_ana[0][0][0], list_ana[0][0][-1],
        list_bio[0][0][0], list_bio[0][0][-1]]
colors=["blue","cornflowerblue",
        "goldenrod", "gold",
        "red", "coral"]
linewidth=2
linestyles=["-", "-", "--", "--", "-.", "-."]
labels=[r"mbt, $\alpha_z=0m$", r"mbt, $\alpha_z=0.05m$",
        r"ana, $\alpha_z=0m$", r"ana, $\alpha_z=0.05m$",
        r"bio, $\alpha_z=0m$", r"bio, $\alpha_z=0.05m$"]

for i, mod in enumerate(models):
    mod.centerline(time=time, color=colors[i], lw=linewidth, linestyle=linestyles[i], label=labels[i])
plt.legend()
plt.xlim(-10, 275)
plt.show()

xpos = 100
for i, mod in enumerate(models):
    mod.transverse(x_position=xpos, time=time, color=colors[i], lw=linewidth, linestyle=linestyles[i], label=labels[i])
plt.legend()
plt.xlim(-20,20)
plt.show()

Regardless, it is difficult and slightly misleading to represent the differences between models as a single value, since differences are